# 팝빌 계좌조회 API 테스트

프로젝트의 `.env`에 LinkID, SecretKey, 사업자번호, 은행코드, 계좌번호를 입력한 뒤 위에서 아래로 실행하세요.

은행 빠른조회 신청 및 **팝빌 테스트 사이트의 계좌등록/정액제 준비**가 필요합니다. 계좌 비밀번호는 은행·팝빌 등록 화면에 입력하며 이 노트북에는 필요하지 않습니다.

기본 환경은 테스트입니다. 출력에는 실제 거래정보가 포함될 수 있으므로 Git에 넣기 전에 **Clear All Outputs**로 지우세요.

[서비스 소개](https://developers.popbill.com/guide/easyfinbank/introduction/easy-intro) · [조회 준비 절차](https://developers.popbill.com/guide/easyfinbank/introduction/check-bank-account)

## 1. 프로젝트 경로 및 패키지 설치
Python 3.10 이상 커널을 선택하세요. 프로젝트/노트북 폴더 또는 저장소 루트에서 실행할 수 있습니다.

In [ ]:
from pathlib import Path
import sys
import subprocess

cwd = Path.cwd().resolve()
candidates = [cwd, *cwd.parents, cwd / "projects" / "popbill-bank-test"]
ROOT = next((p for p in candidates if (p / "src" / "bank_client.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("popbill-bank-test 폴더를 열고 노트북을 실행하세요.")
print("프로젝트:", ROOT)


In [ ]:
# 최초 1회 실행. 현재 선택한 노트북 커널에 설치합니다.
subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", str(ROOT / "requirements.txt")])


## 2. .env 설정 읽기 (API 호출 없음)
`.env.example`과 같은 형식으로 프로젝트의 `.env`를 작성하세요. 날짜를 비워두면 한국시간 어제~오늘을 조회합니다. 값을 수정하면 이 셀부터 다시 실행하세요.

In [ ]:
import importlib
import pandas as pd
from IPython.display import display

if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
import bank_client
importlib.reload(bank_client)
from bank_client import BankClient, load_config

# 설정 변경 시 이전 계좌의 작업/결과를 재사용하지 않습니다.
job_id = None
job_state = None
transactions = None
summary = None
df = None
config = load_config()
client = BankClient(config)
print("환경:", "테스트" if config["is_test"] else "운영")
print("은행코드:", config["bank_code"])
print("계좌:", "****" + config["account_number"][-4:])
print("기간:", config["start_date"], "~", config["end_date"])


## 3. 등록된 계좌 확인 (API 호출)
등록 여부를 확인합니다. 등록되지 않았다면 [팝빌 테스트 사이트](https://test.popbill.com)에서 계좌를 먼저 등록하세요.

In [ ]:
account = client.account_info()
print("등록 계좌정보 조회 성공")
# 인증정보가 포함될 수 있는 계좌정보 전체 대신 상태 항목만 표시합니다.
display({key: account[key] for key in ("bankCode", "accountName", "accountType", "state", "contractState") if key in account})


## 4. 거래내역 수집 요청 (API 호출)
실행할 때마다 새 작업이 만들어집니다. 대기시간이 초과되었을 때는 이 셀 대신 다음 셀만 다시 실행하세요. 작업 ID는 요청 후 1시간 동안 유효합니다.

In [ ]:
job_state = None
transactions = None
summary = None
df = None
job_id = None
job_id = client.request_job()
print("작업 ID:", job_id)


## 5. 수집 완료 대기
상태가 완료(3)이고 결과가 성공(1)인 경우에만 다음 단계로 진행합니다. 대기시간은 `.env`에서 변경할 수 있습니다. 진행 중인 SDK 통신은 SDK 자체 제한시간이 적용되므로 설정한 대기시간을 넘길 수 있습니다.

In [ ]:
if not job_id:
    raise RuntimeError("먼저 수집 요청 셀을 실행하세요.")
job_state = None
job_state = client.wait_for_job(job_id)
print("수집 완료:", job_state["jobState"], "/ 결과:", job_state["errorCode"])


## 6. 거래내역 표와 입출금 합계 조회
전체 페이지를 조회하고 처음 100건을 표시합니다. 금액은 SDK가 반환한 문자열로 보존합니다. `balance`가 제공되면 해당 거래 시점의 잔액이며 현재 잔액과 다를 수 있습니다.

In [ ]:
if not job_state or int(job_state["jobState"]) != 3 or int(job_state["errorCode"]) != 1:
    raise RuntimeError("수집 완료 대기 셀을 먼저 실행하세요.")
transactions = None
summary = None
df = None
transactions = client.transactions(job_id)
summary = client.summary(job_id)
df = pd.DataFrame(transactions)
print(f"조회 건수: {len(df):,}건")
display(pd.DataFrame([summary]))
if df.empty:
    print("조회기간에 거래내역이 없습니다.")
else:
    display(df.head(100))


## 7. 결과 파일 저장 (선택)
`SAVE_RESULTS = True`로 바꾸고 실행하면 프로젝트 `output` 폴더에 JSON을 저장합니다. 이 폴더와 `.env`는 Git에서 제외됩니다.

In [ ]:
import json
from datetime import datetime

SAVE_RESULTS = False
if SAVE_RESULTS:
    if transactions is None or summary is None:
        raise RuntimeError("거래내역 조회 셀을 먼저 실행하세요.")
    output = ROOT / "output"
    output.mkdir(exist_ok=True)
    path = output / f"bank_query_{datetime.now():%Y%m%d_%H%M%S_%f}.json"
    with path.open("x", encoding="utf-8") as f:
        json.dump({"job_id": job_id, "summary": summary, "transactions": transactions}, f, ensure_ascii=False, indent=2)
    print("저장 완료:", path)
else:
    print("파일 저장 생략")
